# Research Direction 4: Outcome Prediction Modeling

**Date**: January 20, 2026  
**Dataset**: MallWorld structured extractions (2,678 dreams)  
**Focus**: What predicts interaction success in MallWorld dreams?

## Research Questions

1. What factors predict interaction success (location, atmosphere, entity presence, sequence position)?
2. Does early success beget later success (momentum effects)?
3. Which features matter most (feature importance ranking)?
4. Do different rules apply in different contexts (escape vs transaction vs navigation)?

In [1]:
# =============================================================================
# SETUP AND DATA LOADING
# =============================================================================

import json
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, pearsonr
import warnings
warnings.filterwarnings('ignore')

# Load data
data_dir = Path('../structured')
json_files = sorted(data_dir.glob('mallworld-*.json'))
print(f"Found {len(json_files)} JSON files")

# Parse all dreams
dreams = []
skipped = 0

def should_skip_extraction(ext):
    """Check if extraction should be skipped (non-dream posts)."""
    if ext.get('is_not_dream_report'):
        return True
    if ext.get('post_type') in ['meta_discussion', 'question', 'theory', 'art', 'other', 'news']:
        return True
    return False

for f in json_files:
    with open(f) as file:
        raw = json.load(file)
        ext = raw.get('extraction', raw)
        
        if should_skip_extraction(ext):
            skipped += 1
            continue
            
        dream_id = f.stem
        dreams.append({
            'dream_id': dream_id,
            'locations': ext.get('locations', []),
            'connections': ext.get('connections', []),
            'interactions': ext.get('interactions', []),
            'entities': ext.get('entities', []),
            'affective_responses': ext.get('affective_responses', []),
            'dream_outcome': ext.get('dream_outcome', {}),
            'raw': ext
        })

print(f"Loaded {len(dreams)} dreams (skipped {skipped} non-dream posts)")

# Build dataframes
locations_data = []
interactions_data = []
entities_data = []

for dream in dreams:
    dream_id = dream['dream_id']
    
    # Build location lookup for this dream
    loc_lookup = {}
    for i, loc in enumerate(dream['locations']):
        loc_id = loc.get('location_id', f'loc_{i}')
        loc_lookup[loc_id] = loc
        locations_data.append({
            'dream_id': dream_id,
            'location_id': loc_id,
            'location_type': loc.get('location_type'),
            'atmosphere': loc.get('atmosphere'),
            'reality_stability': loc.get('reality_stability'),
            'vertical_position': loc.get('vertical_position'),
            'sequence_position': i
        })
    
    # Process interactions with sequence position
    for i, inter in enumerate(dream['interactions']):
        loc_id = inter.get('location_id')
        loc_info = loc_lookup.get(loc_id, {})
        
        interactions_data.append({
            'dream_id': dream_id,
            'interaction_idx': i,
            'location_id': loc_id,
            'interaction_type': inter.get('interaction_type'),
            'outcome': inter.get('outcome'),
            'failure_type': inter.get('failure_type'),
            'entities_involved': inter.get('entities_involved', []),
            'location_type': loc_info.get('location_type'),
            'atmosphere': loc_info.get('atmosphere'),
            'reality_stability': loc_info.get('reality_stability'),
            'vertical_position': loc_info.get('vertical_position')
        })
    
    # Process entities
    for ent in dream.get('entities', []):
        entities_data.append({
            'dream_id': dream_id,
            'entity_type': ent.get('entity_type'),
            'demeanor': ent.get('demeanor'),
            'role': ent.get('role')
        })

df_locations = pd.DataFrame(locations_data)
df_interactions = pd.DataFrame(interactions_data)
df_entities = pd.DataFrame(entities_data)

print(f"\nDataframes created:")
print(f"  Locations: {len(df_locations)}")
print(f"  Interactions: {len(df_interactions)}")
print(f"  Entities: {len(df_entities)}")

Found 3732 JSON files
Loaded 3732 dreams (skipped 0 non-dream posts)

Dataframes created:
  Locations: 12492
  Interactions: 7558
  Entities: 0


## Part 1: Interaction Outcome Overview

In [2]:
# =============================================================================
# OUTCOME DISTRIBUTION ANALYSIS
# =============================================================================

print("="*70)
print("INTERACTION OUTCOME DISTRIBUTION")
print("="*70)

outcome_dist = df_interactions['outcome'].value_counts()
print("\nRaw counts:")
print(outcome_dist)

print("\nPercentages:")
print((outcome_dist / len(df_interactions) * 100).round(1))

# Define outcome categories for modeling
# Binary: succeeded vs not succeeded (for clear prediction)
# Exclude 'not_mentioned' and 'ongoing' as uninformative
meaningful_outcomes = ['succeeded', 'failed', 'interrupted', 'abandoned']
df_meaningful = df_interactions[df_interactions['outcome'].isin(meaningful_outcomes)].copy()
df_meaningful['success'] = (df_meaningful['outcome'] == 'succeeded').astype(int)

print(f"\nMeaningful outcomes for modeling: {len(df_meaningful)} ({len(df_meaningful)/len(df_interactions)*100:.1f}%)")
print(f"Success rate: {df_meaningful['success'].mean()*100:.1f}%")

INTERACTION OUTCOME DISTRIBUTION

Raw counts:
outcome
not_mentioned    2769
succeeded        2082
ongoing          1458
failed            776
interrupted       388
abandoned          85
Name: count, dtype: int64

Percentages:
outcome
not_mentioned    36.6
succeeded        27.5
ongoing          19.3
failed           10.3
interrupted       5.1
abandoned         1.1
Name: count, dtype: float64

Meaningful outcomes for modeling: 3331 (44.1%)
Success rate: 62.5%


In [3]:
# =============================================================================
# SUCCESS RATE BY INTERACTION TYPE
# =============================================================================

print("="*70)
print("SUCCESS RATE BY INTERACTION TYPE")
print("="*70)

type_success = df_meaningful.groupby('interaction_type').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
).sort_values('total', ascending=False)

type_success['success_rate'] = (type_success['success_rate'] * 100).round(1)
print(type_success.head(15))

# Chi-square test for interaction type independence
type_outcome_ct = pd.crosstab(df_meaningful['interaction_type'], df_meaningful['outcome'])
chi2, p, dof, expected = chi2_contingency(type_outcome_ct)
print(f"\nChi-square (type × outcome): χ² = {chi2:.1f}, df = {dof}, p < 0.0001" if p < 0.0001 else f"\nChi-square: χ² = {chi2:.1f}, df = {dof}, p = {p:.4f}")

SUCCESS RATE BY INTERACTION TYPE
                  total  succeeded  success_rate
interaction_type                                
navigation          791        524          66.2
task                554        335          60.5
escape              538        286          53.2
observation         350        299          85.4
social              349        266          76.2
search              300         87          29.0
conflict            205        124          60.5
transaction         168        108          64.3
other                76         53          69.7

Chi-square (type × outcome): χ² = 534.2, df = 24, p < 0.0001


## Part 2: Location and Atmosphere Effects on Success

In [4]:
# =============================================================================
# SUCCESS RATE BY LOCATION TYPE
# =============================================================================

print("="*70)
print("SUCCESS RATE BY LOCATION TYPE")
print("="*70)

# Filter to meaningful outcomes with location type
df_with_loc = df_meaningful[df_meaningful['location_type'].notna()].copy()
print(f"Interactions with location type: {len(df_with_loc)}")

loc_success = df_with_loc.groupby('location_type').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
).sort_values('total', ascending=False)

loc_success['success_rate'] = (loc_success['success_rate'] * 100).round(1)

# Top 15 locations
print("\nTop 15 location types:")
print(loc_success.head(15))

# Chi-square test
loc_outcome_ct = pd.crosstab(df_with_loc['location_type'], df_with_loc['outcome'])
chi2, p, dof, expected = chi2_contingency(loc_outcome_ct)
print(f"\nChi-square (location × outcome): χ² = {chi2:.1f}, df = {dof}, p < 0.0001" if p < 0.0001 else f"\nChi-square: χ² = {chi2:.1f}, df = {dof}, p = {p:.4f}")

SUCCESS RATE BY LOCATION TYPE
Interactions with location type: 3309

Top 15 location types:
                 total  succeeded  success_rate
location_type                                  
other              908        597          65.7
mall               395        236          59.7
mall_store         269        169          62.8
house              148         97          65.5
city_street        112         74          66.1
school              96         52          54.2
restaurant          87         63          72.4
hotel               77         43          55.8
parking_lot         67         38          56.7
hotel_room          49         31          63.3
beach               44         28          63.6
bathroom_public     42         21          50.0
office              40         29          72.5
theater             40         27          67.5
mall_theater        37         27          73.0

Chi-square (location × outcome): χ² = 301.5, df = 207, p < 0.0001


In [8]:
# =============================================================================
# FIX DATA LOADING - Atmosphere is nested in 'qualities'
# =============================================================================

# Rebuild locations with correct atmosphere extraction
locations_data = []
interactions_data = []

for dream in dreams:
    dream_id = dream['dream_id']
    
    # Build location lookup with correct field paths
    loc_lookup = {}
    for i, loc in enumerate(dream['locations']):
        loc_id = loc.get('location_id', f'loc_{i}')
        
        # Get qualities (nested structure)
        qualities = loc.get('qualities', {})
        position = loc.get('position', {})
        
        loc_lookup[loc_id] = {
            'location_type': loc.get('location_type'),
            'atmosphere': qualities.get('atmosphere'),
            'reality_stability': qualities.get('reality_stability'),
            'vertical_position': position.get('vertical')
        }
        
        locations_data.append({
            'dream_id': dream_id,
            'location_id': loc_id,
            'location_type': loc.get('location_type'),
            'atmosphere': qualities.get('atmosphere'),
            'reality_stability': qualities.get('reality_stability'),
            'vertical_position': position.get('vertical'),
            'sequence_position': i
        })
    
    # Process interactions
    for i, inter in enumerate(dream['interactions']):
        loc_id = inter.get('location_id')
        loc_info = loc_lookup.get(loc_id, {})
        
        interactions_data.append({
            'dream_id': dream_id,
            'interaction_idx': i,
            'location_id': loc_id,
            'interaction_type': inter.get('interaction_type'),
            'outcome': inter.get('outcome'),
            'failure_type': inter.get('failure_type'),
            'entities_involved': inter.get('entities_involved', []),
            'location_type': loc_info.get('location_type'),
            'atmosphere': loc_info.get('atmosphere'),
            'reality_stability': loc_info.get('reality_stability'),
            'vertical_position': loc_info.get('vertical_position')
        })

df_locations = pd.DataFrame(locations_data)
df_interactions = pd.DataFrame(interactions_data)

# Recreate meaningful subset
df_meaningful = df_interactions[df_interactions['outcome'].isin(meaningful_outcomes)].copy()
df_meaningful['success'] = (df_meaningful['outcome'] == 'succeeded').astype(int)

print(f"Locations with atmosphere: {df_locations['atmosphere'].notna().sum()} / {len(df_locations)}")
print(f"Interactions with atmosphere: {df_meaningful['atmosphere'].notna().sum()} / {len(df_meaningful)}")
print(f"\nAtmosphere distribution:")
print(df_meaningful['atmosphere'].value_counts())

Locations with atmosphere: 12492 / 12492
Interactions with atmosphere: 3309 / 3331

Atmosphere distribution:
atmosphere
not_mentioned    1029
threatening       503
neutral           366
uncomfortable     328
eerie             303
chaotic           249
welcoming         166
oppressive        159
wrong              94
peaceful           65
nostalgic          47
Name: count, dtype: int64


In [9]:
# =============================================================================
# SUCCESS RATE BY ATMOSPHERE
# =============================================================================

print("="*70)
print("SUCCESS RATE BY ATMOSPHERE")
print("="*70)

# Filter to valid atmosphere values
df_atmos = df_meaningful[
    (df_meaningful['atmosphere'].notna()) & 
    (df_meaningful['atmosphere'] != 'not_mentioned')
].copy()
print(f"Interactions with specified atmosphere: {len(df_atmos)}")

atmos_success = df_atmos.groupby('atmosphere').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
).sort_values('success_rate', ascending=False)

atmos_success['success_rate'] = (atmos_success['success_rate'] * 100).round(1)
print("\nSuccess rate by atmosphere:")
print(atmos_success)

# Group into positive/negative valence
positive_atmos = ['welcoming', 'peaceful', 'nostalgic']
negative_atmos = ['threatening', 'eerie', 'oppressive', 'wrong', 'chaotic', 'uncomfortable']

df_atmos['atmos_valence'] = df_atmos['atmosphere'].apply(
    lambda x: 'positive' if x in positive_atmos else ('negative' if x in negative_atmos else 'neutral')
)

valence_success = df_atmos.groupby('atmos_valence').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
)
valence_success['success_rate'] = (valence_success['success_rate'] * 100).round(1)
print("\n" + "="*70)
print("SUCCESS BY ATMOSPHERE VALENCE")
print("="*70)
print(valence_success)

# Chi-square test for atmosphere effect
from scipy.stats import chi2_contingency
atmos_ct = pd.crosstab(df_atmos['atmos_valence'], df_atmos['success'])
chi2, p, dof, expected = chi2_contingency(atmos_ct)
print(f"\nChi-square (valence × success): χ² = {chi2:.1f}, df = {dof}, p = {p:.6f}")

SUCCESS RATE BY ATMOSPHERE
Interactions with specified atmosphere: 2280

Success rate by atmosphere:
               total  succeeded  success_rate
atmosphere                                   
welcoming        166        141          84.9
peaceful          65         51          78.5
nostalgic         47         34          72.3
neutral          366        259          70.8
eerie            303        202          66.7
chaotic          249        152          61.0
threatening      503        273          54.3
wrong             94         51          54.3
uncomfortable    328        174          53.0
oppressive       159         83          52.2

SUCCESS BY ATMOSPHERE VALENCE
               total  succeeded  success_rate
atmos_valence                                
negative        1636        935          57.2
neutral          366        259          70.8
positive         278        226          81.3

Chi-square (valence × success): χ² = 72.3, df = 2, p = 0.000000


## Part 3: Vertical Position and Reality Stability Effects

In [10]:
# =============================================================================
# SUCCESS RATE BY VERTICAL POSITION
# =============================================================================

print("="*70)
print("SUCCESS RATE BY VERTICAL POSITION")
print("="*70)

# Filter to valid vertical positions
df_vert = df_meaningful[
    (df_meaningful['vertical_position'].notna()) & 
    (df_meaningful['vertical_position'] != 'not_mentioned') &
    (df_meaningful['vertical_position'] != 'varies')
].copy()
print(f"Interactions with specified vertical position: {len(df_vert)}")

vert_success = df_vert.groupby('vertical_position').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
).sort_values('total', ascending=False)

vert_success['success_rate'] = (vert_success['success_rate'] * 100).round(1)
print("\nSuccess rate by vertical position:")
print(vert_success)

# Chi-square test
vert_ct = pd.crosstab(df_vert['vertical_position'], df_vert['success'])
chi2, p, dof, expected = chi2_contingency(vert_ct)
print(f"\nChi-square (vertical × success): χ² = {chi2:.1f}, df = {dof}, p = {p:.4f}")

SUCCESS RATE BY VERTICAL POSITION
Interactions with specified vertical position: 1116

Success rate by vertical position:
                   total  succeeded  success_rate
vertical_position                                
ground               551        375          68.1
upper                219        142          64.8
lower                173        111          64.2
uppermost            110         66          60.0
lowest                63         35          55.6

Chi-square (vertical × success): χ² = 6.0, df = 4, p = 0.2011


In [11]:
# =============================================================================
# SUCCESS RATE BY REALITY STABILITY
# =============================================================================

print("="*70)
print("SUCCESS RATE BY REALITY STABILITY")
print("="*70)

# Filter to valid reality stability
df_real = df_meaningful[
    (df_meaningful['reality_stability'].notna()) & 
    (df_meaningful['reality_stability'] != 'not_mentioned')
].copy()
print(f"Interactions with specified reality stability: {len(df_real)}")

real_success = df_real.groupby('reality_stability').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
).sort_values('total', ascending=False)

real_success['success_rate'] = (real_success['success_rate'] * 100).round(1)
print("\nSuccess rate by reality stability:")
print(real_success)

# Chi-square test
real_ct = pd.crosstab(df_real['reality_stability'], df_real['success'])
chi2, p, dof, expected = chi2_contingency(real_ct)
print(f"\nChi-square (reality × success): χ² = {chi2:.1f}, df = {dof}, p = {p:.4f}")

SUCCESS RATE BY REALITY STABILITY
Interactions with specified reality stability: 1987

Success rate by reality stability:
                   total  succeeded  success_rate
reality_stability                                
solid               1447        954          65.9
shifting             409        235          57.5
hyper_real           106         77          72.6
plastic               18          9          50.0
decaying               7          7         100.0

Chi-square (reality × success): χ² = 18.7, df = 4, p = 0.0009


## Part 4: Entity Presence Effects

In [12]:
# =============================================================================
# ENTITY PRESENCE EFFECTS ON SUCCESS
# =============================================================================

print("="*70)
print("ENTITY PRESENCE EFFECTS ON SUCCESS")
print("="*70)

# Extract entity information from interactions
df_meaningful['has_entities'] = df_meaningful['entities_involved'].apply(
    lambda x: len(x) > 0 if isinstance(x, list) else False
)
df_meaningful['entity_count'] = df_meaningful['entities_involved'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# Success rate by entity presence
entity_success = df_meaningful.groupby('has_entities').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
)
entity_success['success_rate'] = (entity_success['success_rate'] * 100).round(1)
print("Success rate by entity presence:")
print(entity_success)

# Chi-square test
entity_ct = pd.crosstab(df_meaningful['has_entities'], df_meaningful['success'])
chi2, p, dof, expected = chi2_contingency(entity_ct)
print(f"\nChi-square (entity presence × success): χ² = {chi2:.1f}, df = {dof}, p = {p:.4f}")

# Correlation between entity count and success
valid_count = df_meaningful[df_meaningful['entity_count'] > 0]
if len(valid_count) > 10:
    corr, p_corr = spearmanr(valid_count['entity_count'], valid_count['success'])
    print(f"\nSpearman (entity count × success, n={len(valid_count)}): ρ = {corr:.3f}, p = {p_corr:.4f}")

ENTITY PRESENCE EFFECTS ON SUCCESS
Success rate by entity presence:
              total  succeeded  success_rate
has_entities                                
False          1811       1069          59.0
True           1520       1013          66.6

Chi-square (entity presence × success): χ² = 20.1, df = 1, p = 0.0000

Spearman (entity count × success, n=1520): ρ = -0.028, p = 0.2732


In [13]:
# =============================================================================
# SUCCESS BY ENTITY TYPE
# =============================================================================

print("="*70)
print("SUCCESS BY ENTITY TYPE (present in interaction)")
print("="*70)

# Extract entity types from each interaction
entity_type_data = []
for _, row in df_meaningful.iterrows():
    entities = row['entities_involved']
    if isinstance(entities, list):
        for ent in entities:
            if isinstance(ent, dict) and 'entity_type' in ent:
                entity_type_data.append({
                    'dream_id': row['dream_id'],
                    'interaction_idx': row['interaction_idx'],
                    'entity_type': ent['entity_type'],
                    'demeanor': ent.get('demeanor'),
                    'success': row['success']
                })

df_entity_inter = pd.DataFrame(entity_type_data)
print(f"Entity-interaction pairs: {len(df_entity_inter)}")

# Success rate by entity type
ent_type_success = df_entity_inter.groupby('entity_type').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
).sort_values('total', ascending=False)
ent_type_success['success_rate'] = (ent_type_success['success_rate'] * 100).round(1)

print("\nSuccess rate by entity type:")
print(ent_type_success)

# Chi-square test
ent_ct = pd.crosstab(df_entity_inter['entity_type'], df_entity_inter['success'])
chi2, p, dof, expected = chi2_contingency(ent_ct)
print(f"\nChi-square (entity type × success): χ² = {chi2:.1f}, df = {dof}, p = {p:.4f}")

SUCCESS BY ENTITY TYPE (present in interaction)
Entity-interaction pairs: 1865

Success rate by entity type:
               total  succeeded  success_rate
entity_type                                  
stranger         497        359          72.2
crowd            311        207          66.6
authority        286        163          57.0
threat           169        105          62.1
known_person     131         88          67.2
family_member    116         68          58.6
other            105         69          65.7
creature          88         55          62.5
child             45         35          77.8
friend            44         30          68.2
guide             27         24          88.9
deceased          25         18          72.0
coworker           8          6          75.0
shadow             6          5          83.3
none               3          2          66.7
watcher            3          1          33.3
faceless           1          0           0.0

Chi-square (enti

In [14]:
# =============================================================================
# SUCCESS BY ENTITY DEMEANOR
# =============================================================================

print("="*70)
print("SUCCESS BY ENTITY DEMEANOR")
print("="*70)

# Filter to valid demeanors
df_demeanor = df_entity_inter[
    (df_entity_inter['demeanor'].notna()) & 
    (df_entity_inter['demeanor'] != 'not_mentioned')
]

dem_success = df_demeanor.groupby('demeanor').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
).sort_values('success_rate', ascending=False)
dem_success['success_rate'] = (dem_success['success_rate'] * 100).round(1)

print("Success rate by entity demeanor:")
print(dem_success)

# Group by valence
positive_dem = ['friendly', 'helpful']
negative_dem = ['hostile', 'threatening', 'unfriendly']

df_demeanor = df_demeanor.copy()
df_demeanor['dem_valence'] = df_demeanor['demeanor'].apply(
    lambda x: 'positive' if x in positive_dem else ('negative' if x in negative_dem else 'neutral')
)

dem_valence_success = df_demeanor.groupby('dem_valence').agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
)
dem_valence_success['success_rate'] = (dem_valence_success['success_rate'] * 100).round(1)

print("\nSuccess by demeanor valence:")
print(dem_valence_success)

# Chi-square
dem_ct = pd.crosstab(df_demeanor['dem_valence'], df_demeanor['success'])
chi2, p, dof, expected = chi2_contingency(dem_ct)
print(f"\nChi-square (demeanor valence × success): χ² = {chi2:.1f}, df = {dof}, p = {p:.4f}")

SUCCESS BY ENTITY DEMEANOR
Success rate by entity demeanor:
             total  succeeded  success_rate
demeanor                                   
friendly       227        202          89.0
helpful        100         87          87.0
neutral        483        363          75.2
watching        39         25          64.1
threatening    142         83          58.5
hostile        254        145          57.1
confusing      120         61          50.8
unfriendly     155         69          44.5
indifferent     66         17          25.8

Success by demeanor valence:
             total  succeeded  success_rate
dem_valence                                
negative       551        297          53.9
neutral        708        466          65.8
positive       327        289          88.4

Chi-square (demeanor valence × success): χ² = 109.4, df = 2, p = 0.0000


## Part 5: Sequence Position and Momentum Effects

In [15]:
# =============================================================================
# SUCCESS BY SEQUENCE POSITION
# =============================================================================

print("="*70)
print("SUCCESS BY SEQUENCE POSITION (interaction order in dream)")
print("="*70)

# Analyze success by interaction position
df_meaningful['position_category'] = pd.cut(
    df_meaningful['interaction_idx'], 
    bins=[-1, 0, 1, 2, 4, 100],
    labels=['First', 'Second', 'Third', '4-5th', '6th+']
)

pos_success = df_meaningful.groupby('position_category', observed=True).agg(
    total=('success', 'count'),
    succeeded=('success', 'sum'),
    success_rate=('success', 'mean')
)
pos_success['success_rate'] = (pos_success['success_rate'] * 100).round(1)

print("Success rate by sequence position:")
print(pos_success)

# Correlation between raw position and success
corr, p_corr = spearmanr(df_meaningful['interaction_idx'], df_meaningful['success'])
print(f"\nSpearman (interaction position × success): ρ = {corr:.3f}, p = {p_corr:.4f}")

SUCCESS BY SEQUENCE POSITION (interaction order in dream)
Success rate by sequence position:
                   total  succeeded  success_rate
position_category                                
First                827        529          64.0
Second               801        486          60.7
Third                582        342          58.8
4-5th                628        412          65.6
6th+                 493        313          63.5

Spearman (interaction position × success): ρ = 0.008, p = 0.6440


In [16]:
# =============================================================================
# MOMENTUM ANALYSIS: Does prior success predict future success?
# =============================================================================

print("="*70)
print("MOMENTUM ANALYSIS: Prior success effect on current success")
print("="*70)

# For each interaction, compute prior success rate in the same dream
momentum_data = []

for dream_id, group in df_meaningful.groupby('dream_id'):
    group = group.sort_values('interaction_idx')
    outcomes = group['success'].tolist()
    
    for i, (_, row) in enumerate(group.iterrows()):
        prior_successes = sum(outcomes[:i])
        prior_total = i
        
        momentum_data.append({
            'dream_id': dream_id,
            'interaction_idx': row['interaction_idx'],
            'success': row['success'],
            'prior_successes': prior_successes,
            'prior_total': prior_total,
            'prior_success_rate': prior_successes / prior_total if prior_total > 0 else np.nan
        })

df_momentum = pd.DataFrame(momentum_data)

# Filter to interactions that have prior context
df_with_prior = df_momentum[df_momentum['prior_total'] > 0]
print(f"Interactions with prior context: {len(df_with_prior)}")

# Correlation between prior success and current success
corr, p_corr = spearmanr(df_with_prior['prior_success_rate'], df_with_prior['success'])
print(f"\nSpearman (prior success rate × current success): ρ = {corr:.3f}, p = {p_corr:.4f}")

# Compare success rate after success vs after failure (first prior outcome only)
df_second_plus = df_momentum[df_momentum['prior_total'] >= 1].copy()
df_second_plus['first_outcome'] = df_second_plus['prior_successes'].apply(
    lambda x: 'after_success' if x >= 1 else 'after_failure'
)

first_momentum = df_second_plus.groupby('first_outcome').agg(
    total=('success', 'count'),
    success_rate=('success', 'mean')
)
first_momentum['success_rate'] = (first_momentum['success_rate'] * 100).round(1)
print("\nSuccess rate based on at least one prior success:")
print(first_momentum)

# Chi-square
mom_ct = pd.crosstab(df_second_plus['first_outcome'], df_second_plus['success'])
chi2, p, dof, _ = chi2_contingency(mom_ct)
print(f"\nChi-square: χ² = {chi2:.1f}, df = {dof}, p = {p:.4f}")

MOMENTUM ANALYSIS: Prior success effect on current success
Interactions with prior context: 1852

Spearman (prior success rate × current success): ρ = 0.204, p = 0.0000

Success rate based on at least one prior success:
               total  success_rate
first_outcome                     
after_failure    448          48.4
after_success   1404          68.4

Chi-square: χ² = 57.4, df = 1, p = 0.0000


## Part 6: Multivariate Prediction Models

In [17]:
# =============================================================================
# PREPARE DATA FOR LOGISTIC REGRESSION
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, roc_auc_score

print("="*70)
print("BUILDING MULTIVARIATE PREDICTION MODEL")
print("="*70)

# Create feature matrix
# First, prepare all the needed features
df_model = df_meaningful.copy()

# Atmosphere valence (encode: positive=2, neutral=1, negative=0)
positive_atmos = ['welcoming', 'peaceful', 'nostalgic']
negative_atmos = ['threatening', 'eerie', 'oppressive', 'wrong', 'chaotic', 'uncomfortable']
df_model['atmos_valence'] = df_model['atmosphere'].apply(
    lambda x: 2 if x in positive_atmos else (0 if x in negative_atmos else 1)
)

# Entity presence (binary)
df_model['has_entities'] = df_model['entities_involved'].apply(
    lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0
)

# Interaction type (one-hot encode top types)
top_types = ['navigation', 'task', 'escape', 'observation', 'social', 'search', 'conflict', 'transaction']
for t in top_types:
    df_model[f'type_{t}'] = (df_model['interaction_type'] == t).astype(int)

# Reality stability (solid=1, shifting=0)
df_model['reality_solid'] = (df_model['reality_stability'] == 'solid').astype(int)

# Sequence position (early=0, mid=1, late=2)
df_model['seq_position'] = pd.cut(
    df_model['interaction_idx'], 
    bins=[-1, 1, 3, 100],
    labels=[0, 1, 2]
).astype(int)

# Select features for model
feature_cols = ['atmos_valence', 'has_entities', 'reality_solid', 'seq_position'] + [f'type_{t}' for t in top_types]
df_model_clean = df_model.dropna(subset=feature_cols + ['success'])

X = df_model_clean[feature_cols].values
y = df_model_clean['success'].values

print(f"Model dataset: {len(df_model_clean)} interactions")
print(f"Features: {len(feature_cols)}")
print(f"Success rate: {y.mean()*100:.1f}%")

BUILDING MULTIVARIATE PREDICTION MODEL
Model dataset: 3331 interactions
Features: 12
Success rate: 62.5%


In [18]:
# =============================================================================
# LOGISTIC REGRESSION MODEL
# =============================================================================

print("="*70)
print("LOGISTIC REGRESSION MODEL")
print("="*70)

# Fit logistic regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X, y)

# Cross-validation score
cv_scores = cross_val_score(lr_model, X, y, cv=5, scoring='roc_auc')
print(f"\n5-Fold Cross-Validation ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")

# Feature coefficients (odds ratios)
print("\nFeature Coefficients (Odds Ratios):")
print("-" * 50)

coefs = list(zip(feature_cols, lr_model.coef_[0]))
coefs_sorted = sorted(coefs, key=lambda x: abs(x[1]), reverse=True)

for feat, coef in coefs_sorted:
    odds_ratio = np.exp(coef)
    direction = "↑" if coef > 0 else "↓"
    print(f"  {feat:20s}: OR = {odds_ratio:.3f} {direction}")

# Overall model accuracy
y_pred = lr_model.predict(X)
accuracy = (y_pred == y).mean()
print(f"\nModel accuracy: {accuracy*100:.1f}%")
print(f"Baseline (always predict success): {y.mean()*100:.1f}%")

LOGISTIC REGRESSION MODEL

5-Fold Cross-Validation ROC-AUC: 0.671 (+/- 0.043)

Feature Coefficients (Odds Ratios):
--------------------------------------------------
  type_search         : OR = 0.220 ↓
  type_observation    : OR = 2.881 ↑
  type_escape         : OR = 0.632 ↓
  atmos_valence       : OR = 1.510 ↑
  has_entities        : OR = 1.325 ↑
  type_task           : OR = 0.767 ↓
  type_conflict       : OR = 0.767 ↓
  reality_solid       : OR = 1.255 ↑
  type_social         : OR = 1.234 ↑
  type_transaction    : OR = 0.815 ↓
  seq_position        : OR = 1.052 ↑
  type_navigation     : OR = 1.027 ↑

Model accuracy: 66.9%
Baseline (always predict success): 62.5%


In [19]:
# =============================================================================
# RANDOM FOREST MODEL AND FEATURE IMPORTANCE
# =============================================================================

print("="*70)
print("RANDOM FOREST MODEL AND FEATURE IMPORTANCE")
print("="*70)

# Fit random forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X, y)

# Cross-validation score
rf_cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='roc_auc')
print(f"\n5-Fold Cross-Validation ROC-AUC: {rf_cv_scores.mean():.3f} (+/- {rf_cv_scores.std()*2:.3f})")

# Feature importance
print("\nFeature Importance (Mean Decrease in Impurity):")
print("-" * 50)

importances = list(zip(feature_cols, rf_model.feature_importances_))
importances_sorted = sorted(importances, key=lambda x: x[1], reverse=True)

for feat, imp in importances_sorted:
    bar = "█" * int(imp * 50)
    print(f"  {feat:20s}: {imp:.3f} {bar}")

# Overall model accuracy
y_pred_rf = rf_model.predict(X)
accuracy_rf = (y_pred_rf == y).mean()
print(f"\nModel accuracy: {accuracy_rf*100:.1f}%")

RANDOM FOREST MODEL AND FEATURE IMPORTANCE

5-Fold Cross-Validation ROC-AUC: 0.645 (+/- 0.042)

Feature Importance (Mean Decrease in Impurity):
--------------------------------------------------
  type_search         : 0.226 ███████████
  atmos_valence       : 0.168 ████████
  seq_position        : 0.166 ████████
  type_observation    : 0.119 █████
  reality_solid       : 0.085 ████
  has_entities        : 0.075 ███
  type_escape         : 0.041 ██
  type_social         : 0.033 █
  type_navigation     : 0.029 █
  type_task           : 0.026 █
  type_transaction    : 0.017 
  type_conflict       : 0.015 

Model accuracy: 68.6%


## Part 7: Context-Specific Success Predictors

In [20]:
# =============================================================================
# ESCAPE-SPECIFIC SUCCESS PREDICTORS
# =============================================================================

print("="*70)
print("ESCAPE INTERACTION SUCCESS PREDICTORS")
print("="*70)

df_escape = df_meaningful[df_meaningful['interaction_type'] == 'escape'].copy()
print(f"Escape interactions: {len(df_escape)}")
print(f"Escape success rate: {df_escape['success'].mean()*100:.1f}%")

# Success by atmosphere for escape
escape_atmos = df_escape[df_escape['atmosphere'].notna() & (df_escape['atmosphere'] != 'not_mentioned')]
if len(escape_atmos) > 0:
    escape_atmos_success = escape_atmos.groupby('atmosphere').agg(
        total=('success', 'count'),
        success_rate=('success', 'mean')
    ).sort_values('total', ascending=False)
    escape_atmos_success['success_rate'] = (escape_atmos_success['success_rate'] * 100).round(1)
    print("\nEscape success by atmosphere:")
    print(escape_atmos_success.head(10))

# Success by entity presence for escape
print("\nEscape success by entity presence:")
df_escape['has_entities'] = df_escape['entities_involved'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
escape_entity = df_escape.groupby('has_entities').agg(
    total=('success', 'count'),
    success_rate=('success', 'mean')
)
escape_entity['success_rate'] = (escape_entity['success_rate'] * 100).round(1)
print(escape_entity)

ESCAPE INTERACTION SUCCESS PREDICTORS
Escape interactions: 538
Escape success rate: 53.2%

Escape success by atmosphere:
               total  success_rate
atmosphere                        
threatening      219          50.7
oppressive        50          44.0
eerie             47          48.9
uncomfortable     46          52.2
chaotic           44          70.5
neutral           23          56.5
wrong             11          54.5
peaceful           7          57.1
welcoming          6         100.0

Escape success by entity presence:
              total  success_rate
has_entities                     
False           263          47.5
True            275          58.5


In [21]:
# =============================================================================
# SEARCH-SPECIFIC SUCCESS PREDICTORS (lowest success rate)
# =============================================================================

print("="*70)
print("SEARCH INTERACTION SUCCESS PREDICTORS")
print("="*70)

df_search = df_meaningful[df_meaningful['interaction_type'] == 'search'].copy()
print(f"Search interactions: {len(df_search)}")
print(f"Search success rate: {df_search['success'].mean()*100:.1f}%")

# Success by location type for search
search_loc = df_search[df_search['location_type'].notna()]
if len(search_loc) > 0:
    search_loc_success = search_loc.groupby('location_type').agg(
        total=('success', 'count'),
        success_rate=('success', 'mean')
    ).sort_values('total', ascending=False)
    search_loc_success['success_rate'] = (search_loc_success['success_rate'] * 100).round(1)
    print("\nSearch success by location type:")
    print(search_loc_success.head(10))

# Success by atmosphere for search
search_atmos = df_search[df_search['atmosphere'].notna() & (df_search['atmosphere'] != 'not_mentioned')]
if len(search_atmos) > 0:
    search_atmos_success = search_atmos.groupby('atmosphere').agg(
        total=('success', 'count'),
        success_rate=('success', 'mean')
    ).sort_values('total', ascending=False)
    search_atmos_success['success_rate'] = (search_atmos_success['success_rate'] * 100).round(1)
    print("\nSearch success by atmosphere:")
    print(search_atmos_success.head(10))

SEARCH INTERACTION SUCCESS PREDICTORS
Search interactions: 300
Search success rate: 29.0%

Search success by location type:
                 total  success_rate
location_type                       
mall                46          26.1
other               43          27.9
mall_store          34          29.4
parking_lot         18          27.8
school              17          11.8
house               12          66.7
hotel               10          30.0
bathroom_public      8          12.5
city_street          7          57.1
restaurant           7          57.1

Search success by atmosphere:
               total  success_rate
atmosphere                        
uncomfortable     45          22.2
neutral           34          41.2
chaotic           33          27.3
eerie             28          39.3
threatening       21          19.0
wrong             18          16.7
oppressive         9          11.1
welcoming          8          37.5
nostalgic          5          40.0
peaceful        

In [22]:
# =============================================================================
# OBSERVATION SUCCESS PREDICTORS (highest success rate)
# =============================================================================

print("="*70)
print("OBSERVATION INTERACTION SUCCESS PREDICTORS")
print("="*70)

df_observe = df_meaningful[df_meaningful['interaction_type'] == 'observation'].copy()
print(f"Observation interactions: {len(df_observe)}")
print(f"Observation success rate: {df_observe['success'].mean()*100:.1f}%")

# Check what makes observations fail
df_observe_fail = df_observe[df_observe['success'] == 0]
print(f"\nFailed observations: {len(df_observe_fail)}")
print("\nFailure types for observations:")
print(df_observe_fail['failure_type'].value_counts())

# Success by atmosphere for observation
observe_atmos = df_observe[df_observe['atmosphere'].notna() & (df_observe['atmosphere'] != 'not_mentioned')]
if len(observe_atmos) > 0:
    observe_atmos_success = observe_atmos.groupby('atmosphere').agg(
        total=('success', 'count'),
        success_rate=('success', 'mean')
    ).sort_values('total', ascending=False)
    observe_atmos_success['success_rate'] = (observe_atmos_success['success_rate'] * 100).round(1)
    print("\nObservation success by atmosphere:")
    print(observe_atmos_success.head(10))

OBSERVATION INTERACTION SUCCESS PREDICTORS
Observation interactions: 350
Observation success rate: 85.4%

Failed observations: 51

Failure types for observations:
failure_type
not_applicable           41
external_intervention     3
no_effect                 3
unclear                   2
environmental_block       2
Name: count, dtype: int64

Observation success by atmosphere:
               total  success_rate
atmosphere                        
eerie             49          83.7
neutral           41          87.8
threatening       35          80.0
uncomfortable     30          80.0
chaotic           16          87.5
welcoming         15          80.0
oppressive        12         100.0
peaceful          12          83.3
wrong             12          91.7
nostalgic         10          90.0


---
## Summary: The Rules of MallWorld Success

In [23]:
print("="*80)
print("COMPREHENSIVE SUMMARY: OUTCOME PREDICTION IN MALLWORLD DREAMS")
print("="*80)

print("""
DATASET OVERVIEW
----------------
- Total interactions analyzed: 7,558
- Meaningful outcomes (success/fail): 3,331 (44.1%)
- Baseline success rate: 62.5%

KEY PREDICTORS OF SUCCESS
=========================

1. INTERACTION TYPE (strongest predictor)
   - Observation: 85.4% success (passive witnessing succeeds)
   - Social: 76.2% success (social interactions favorable)
   - Navigation: 66.2% success (getting around works)
   - Transaction: 64.3% success (buying possible)
   - Task: 60.5% success (chores harder)
   - Escape: 53.2% success (fleeing difficult)
   - Search: 29.0% success (finding things nearly impossible)
   
   Chi-square: χ² = 534.2, df = 24, p < 0.0001

2. ATMOSPHERE VALENCE (second strongest)
   - Positive (welcoming/peaceful/nostalgic): 81.3% success
   - Neutral: 70.8% success  
   - Negative (threatening/eerie/oppressive/etc): 57.2% success
   
   Chi-square: χ² = 72.3, df = 2, p < 0.0001
   Logistic regression OR = 1.51 per valence step

3. ENTITY DEMEANOR (highly significant)
   - Friendly entities: 89.0% success
   - Helpful entities: 87.0% success
   - Neutral entities: 75.2% success
   - Hostile/threatening: 57-58% success
   - Indifferent entities: 25.8% success (worst!)
   
   Chi-square: χ² = 109.4, df = 2, p < 0.0001

4. MOMENTUM EFFECT (success begets success)
   - After prior success: 68.4% success rate
   - After prior failure: 48.4% success rate
   
   Spearman: ρ = 0.204, p < 0.0001
   Chi-square: χ² = 57.4, df = 1, p < 0.0001

5. ENTITY PRESENCE (significant)
   - With entities: 66.6% success
   - Without entities: 59.0% success
   
   Chi-square: χ² = 20.1, df = 1, p < 0.0001

6. REALITY STABILITY (significant)
   - Hyper-real: 72.6% success
   - Solid: 65.9% success
   - Shifting: 57.5% success
   
   Chi-square: χ² = 18.7, df = 4, p = 0.0009

7. VERTICAL POSITION (NOT significant)
   - Ground: 68.1% success
   - Lowest: 55.6% success
   
   Chi-square: χ² = 6.0, df = 4, p = 0.20

PREDICTIVE MODEL PERFORMANCE
============================
- Logistic Regression: ROC-AUC = 0.671
- Random Forest: ROC-AUC = 0.645
- Model accuracy: 66.9-68.6%
- Improvement over baseline: +4.4-6.1%

TOP FEATURE IMPORTANCES (Random Forest):
1. type_search (0.226) - searching is inherently hard
2. atmos_valence (0.168) - atmosphere matters
3. seq_position (0.166) - timing matters
4. type_observation (0.119) - observing is easy
5. reality_solid (0.085) - stability helps

CONTEXT-SPECIFIC RULES
======================

ESCAPE SUCCESS (53.2% overall):
- Chaotic atmosphere: 70.5% (chaos helps escape!)
- With entities present: 58.5% (can get help/distraction)
- Without entities: 47.5%

SEARCH SUCCESS (29.0% overall - very low):
- In house: 66.7% (familiar space helps)
- In school: 11.8% (institutional spaces hostile)
- Neutral atmosphere: 41.2%
- Wrong/oppressive: 11-16%

OBSERVATION (85.4% - highest):
- Success rate stable across atmospheres (80-100%)
- Failure mainly "not_applicable" (passive interaction)

THE RULES OF MALLWORLD
======================

1. WATCH, DON'T SEARCH
   Observation succeeds 3x more than search.
   The mall rewards passive attention, punishes active seeking.

2. ATMOSPHERE IS DESTINY
   Positive atmosphere = 81% success
   Negative atmosphere = 57% success
   A 24 percentage point swing based on environmental mood.

3. SUCCESS BREEDS SUCCESS
   Prior success predicts +20% current success.
   Momentum is real in dream logic.

4. INDIFFERENCE IS WORSE THAN HOSTILITY
   Indifferent entities: 25.8% success
   Hostile entities: 57.1% success
   Being ignored is worse than being attacked.

5. CHAOS ENABLES ESCAPE
   Chaotic atmosphere: 70.5% escape success
   Threatening atmosphere: 50.7% escape success
   Disorder opens exit routes.

6. REALITY ANCHORS ACTION
   Solid reality: 66% success
   Shifting reality: 58% success
   Stable environments enable agency.
""")

print("="*80)
print("ANALYSIS COMPLETE")
print("="*80)

COMPREHENSIVE SUMMARY: OUTCOME PREDICTION IN MALLWORLD DREAMS

DATASET OVERVIEW
----------------
- Total interactions analyzed: 7,558
- Meaningful outcomes (success/fail): 3,331 (44.1%)
- Baseline success rate: 62.5%

KEY PREDICTORS OF SUCCESS

1. INTERACTION TYPE (strongest predictor)
   - Observation: 85.4% success (passive witnessing succeeds)
   - Social: 76.2% success (social interactions favorable)
   - Navigation: 66.2% success (getting around works)
   - Transaction: 64.3% success (buying possible)
   - Task: 60.5% success (chores harder)
   - Escape: 53.2% success (fleeing difficult)
   - Search: 29.0% success (finding things nearly impossible)

   Chi-square: χ² = 534.2, df = 24, p < 0.0001

2. ATMOSPHERE VALENCE (second strongest)
   - Positive (welcoming/peaceful/nostalgic): 81.3% success
   - Neutral: 70.8% success  
   - Negative (threatening/eerie/oppressive/etc): 57.2% success

   Chi-square: χ² = 72.3, df = 2, p < 0.0001
   Logistic regression OR = 1.51 per valence ste